# Analyse Helios Simulations
This notebook walks through the process of analysing helios simulations which follows 3 main steps:

    (1) Data Preparation
        This stage will setup the project directory, setup expected schemas for dataframes (both dask and pandas), and ultimately read in the helios data and prepare the required per ray information into a .parquet output.
        It will also setup the reference dataset for voxels for each voxel_size in the project (i.e. unique voxel_ids etc.).
    
    (2) Voxel-Ray Intersection
        With valid rays saved per leg of the scan, in the previous step, the goal now is to check ray intersections in all voxels. This will record important information, such as the entry/exit/hit coordinates of the ray which will later be used to gather metrics.
        The main reason these metrics are not gathered yet, is that this stage will remain separate per leg. That way, the metrics can be computed from different combinations of helios legs without re-computing voxel-ray intersections.

    (3) Compute Metrics
        Taking a given set of legs and voxel_sizes, the voxel_ray intersection files will be used to calculate metrics for each voxel, in this case resulting in all outputs from each investigated method.

## Step 1 - Data Preparation
This step focuses on converting helios simulation outputs, saving only valid rays into a more efficient .parquet file format.

It expects the following input and will add a new folder (valid_rays) to store all resulting .parquet files.

INPUT:
    project_dir/
    ├── reference/
    │   ├── "{project}_voxel_size_0.2.csv"
    │   ├── "{project}_voxel_size_0.5.csv"
    │   ...
    │   └── "{project}_voxel_size_{v}.csv"
    ├── helios/
    │   ├── "leg000_points.xyz"
    │   ├── "leg000_pulse.txt"
    │   ├── "leg000_fullwave.txt"
    │   ├── "leg001_points.xyz"
    │   ├── "leg001_pulse.txt"
    │   ├── "leg001_fullwave.txt"
    │   ├── ...
    │   ├── "leg{l}_points.xyz"
    │   ├── "leg{l}_pulse.txt"
    │   └── "leg{l}_fullwave.txt"

OUTPUT:
    └── valid_rays/
        ├── "leg_000_valid_rays.parquet"
        ├── "leg_001_valid_rays.parquet"
        ...
        └── "leg_{l}_valid_rays.parquet"

In [ ]:
import os
import utils

# Set up the project directory
project_dir = '/path/to/project/'
helios_dir = os.path.join(project_dir, 'helios')
references_dir = os.path.join(project_dir, 'references')

if not os.path.exists(helios_dir) or not os.path.exists(references_dir):
    raise FileNotFoundError("The specified directories do not exist. Please check the paths.")

valid_rays_dir = os.path.join(project_dir, 'valid_rays')
if not os.path.exists(valid_rays_dir):
    os.makedirs(valid_rays_dir)


# Establish the object IDs for the leaves
leaf_object_ids = [1]

# Run the data preparation script
utils.prepare_helios_data(input_dir=helios_dir, output_dir=valid_rays_dir, references_dir=references_dir, leaf_object_ids=leaf_object_ids)

## Step 2 - Voxel Ray Intersections
This code uses the valid rays from before, alongside the reference datasets in order to create a supporting parquet in the valid rays folder using the voxel_size_{voxel_size}_leg_{leg}_intersections.parquet format.

In [ ]:
import os
import utils

# Set up the project directory
project_dir = '/home/capheus/projects/helios_density_test/200_leaf_60_JoshOutput'
valid_rays_dir = os.path.join(project_dir, 'valid_rays')
references_dir = os.path.join(project_dir, 'references')
temp_dir = os.path.join(project_dir, 'tmp')
if not os.path.exists(temp_dir):
    os.makedirs(temp_dir, exist_ok=True)

### HPC BUNYA ONLY ###
temp_dir = os.getenv('TMPDIR', temp_dir)

# None will calculate from os and psutil.
# Set if you are using a system that doesn't respond too well to this automated method
cpus=None
mem=None    # 1TB is 1024**4, 1GB is 1024**3, 1MB is 1024**2

# Run intersections
utils.voxel_ray_intersections(valid_rays_dir=valid_rays_dir, references_dir=references_dir, cpus=cpus, mem=mem, temp_dir=temp_dir) #debug=True, cpus=None, mem=None Specify here if you want to avoid os.cpu_count()

Processing leg 4...
[                                        ] | 0% Completed | 506.58 msProcess 3ee65a688e0641f288b028fbc4b4de02: Start 518951 rays, 184 voxels, in (6) chunks.
Process 8a036d4215d14d5aa67dcfae042841c7: Start 518951 rays, 184 voxels, in (6) chunks.
Process 7faa2a059acf48e59754413e9118748c: Start 518952 rays, 184 voxels, in (6) chunks.
Process 759ca55bf94e4b6d9326b58b86e7613f: Start 518951 rays, 184 voxels, in (6) chunks.
Process 6848fc51b2b445018a8956ee099ad4db: Start 518951 rays, 184 voxels, in (6) chunks.
Process b4d2eaa870a84225a912ca5c0deac768: Start 518951 rays, 184 voxels, in (6) chunks.
Process 810a3e7d11ce4b1ca0fdfc01ef0b4a2f: Start 518952 rays, 184 voxels, in (6) chunks.
Process 5b6a2d61de8a422288f266d230f96d3d: Start 518952 rays, 184 voxels, in (6) chunks.
[####################                    ] | 51% Completed | 611.13 msProcess 46195a53f689426f85a27f487b22dd53: Start 518952 rays, 184 voxels, in (6) chunks.
Process 67561602690c483fb7059f983f86aecc: Start 5

KeyboardInterrupt: 

Process 9499cce0180b414f9dda12618804d4f6: Finished 518951 rays, 184 voxels. Concatenating results...
Process dae97526434c4a3eb146d7a8f5b83f93: Finished 518952 rays, 184 voxels. Concatenating results...
Process 5ef6c5383d76461898d91bc5975daca3: Finished 518951 rays, 184 voxels. Concatenating results...
Process b5d6d0dc590f45e3b78f02eb844d34cf: Finished 518952 rays, 184 voxels. Concatenating results...
Process 41ec042a9a274cf9997f98673a93fbbc: Finished 518951 rays, 184 voxels. Concatenating results...
Process f0fa8715f8c340fc91b59ba43b4db959: Finished 518951 rays, 184 voxels. Concatenating results...
Process 27b99af16b734e2fb886f0e6f387fda2: Finished 518951 rays, 184 voxels. Concatenating results...
Process 441dad6370784e55b118d00ec6834fb7: Finished 518951 rays, 184 voxels. Concatenating results...
Process ee69e231fd074160b916224c29e96e87: Finished 518952 rays, 184 voxels. Concatenating results...
Process 29aa126d2c7e4e71a5061932a30984ed: Finished 518952 rays, 184 voxels. Concatenating r

# OPTIONAL - Step 2.5 - Add normals and weights to points
Calculating normals and weights (both based on nearest neighbour searches) at the voxel scale introduces issues of (1) unnecessary point sparsity, and (2) unclear planes due to insufficient leaf area. To accomodate for this, we use the Vicari method of calculating leaf inclination angle distribution *on the entire canopy*, from which we can subsequently voxelised pre-computed normals into their correct voxels.

Future versions of this project will incorporate this process in **Step 1**; however, this code block will update *_intersections.parquets which don't include normals to work for the following **Step 3**.

In [ ]:
import numpy as np
import pandas as pd
import glob
import utils
import os
import shutil

# Set up the project directory
project_dir = '/home/capheus/projects/024_Trees'
valid_rays_dir = os.path.join(project_dir, 'valid_rays')

# Establish the voxel_sizes you want to correct
voxel_sizes = [0.2, 0.5, 1.0, 2.0, 1.5] 

for voxel_size in voxel_sizes:
    intersection_files = glob.glob(os.path.join(valid_rays_dir, f'*{voxel_size}_intersections.parquet'))
    if intersection_files == []:
        raise FileNotFoundError("No intersection files found. Please check the directory.")
    
    # Copy the old intersection files to avoid data loss
    for file in intersection_files:
        new_file = file.replace('.parquet', '_old.parquet')
        shutil.copy(file, new_file)

    # Add normals and weights from the intersection files
    new_intersections_df = utils.add_normals_weights_from_intersection_files(files=intersection_files, knn=6)

    # Function to save the new intersection files
    def save_new_file(df):
        leg_id = df.name
        output_filename = os.path.join(valid_rays_dir, f"leg_{leg_id}_voxel_{voxel_size}_intersections.parquet")
        df.to_parquet(output_filename, engine='pyarrow', compression='snappy', schema=utils.voxel_ray_intersection_schema)
        print(f"Saved new intersection file for leg {leg_id} at {output_filename}")
    
    new_intersections_df.groupby('leg_id').apply(lambda x: save_new_file(x))

print("Normals and weights added to the intersection files.")

Adding normals and weights to 12 files...
Saved new intersection file for leg 0 at /home/capheus/projects/024_Trees/valid_rays/leg_0_voxel_1.5_intersections.parquet
Saved new intersection file for leg 1 at /home/capheus/projects/024_Trees/valid_rays/leg_1_voxel_1.5_intersections.parquet
Saved new intersection file for leg 2 at /home/capheus/projects/024_Trees/valid_rays/leg_2_voxel_1.5_intersections.parquet
Saved new intersection file for leg 3 at /home/capheus/projects/024_Trees/valid_rays/leg_3_voxel_1.5_intersections.parquet
Saved new intersection file for leg 4 at /home/capheus/projects/024_Trees/valid_rays/leg_4_voxel_1.5_intersections.parquet
Saved new intersection file for leg 5 at /home/capheus/projects/024_Trees/valid_rays/leg_5_voxel_1.5_intersections.parquet
Saved new intersection file for leg 6 at /home/capheus/projects/024_Trees/valid_rays/leg_6_voxel_1.5_intersections.parquet
Saved new intersection file for leg 7 at /home/capheus/projects/024_Trees/valid_rays/leg_7_voxel_

/tmp/ipykernel_12856/817537203.py:35: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  new_intersections_df.groupby('leg_id').apply(lambda x: save_new_file(x))


## Step 3 - Compute Metrics
Using the leg_{leg_id}_voxel_size_{voxel_size}_intersections.parquet files (which feature a standardised structure of columns from various inputs), compute the desired metrics and save outputs.

In [ ]:
import os
import glob
import utils
import pandas as pd

# Select the desired legs and voxel_sizes to include in the analysis
# Use the shortcut string 'all' to include all 
legs = 'all' # [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11] 
voxel_sizes = [1.5] # 'all' # [0.2, 0.5, 1.0, 2.0]

# Set the average leaf area
average_leaf_area = 0.02  # in m^2, adjust as needed

# Set up the project directory
project_dir = '/home/capheus/projects/024_Trees'
valid_rays_dir = os.path.join(project_dir, 'valid_rays')
references_dir = os.path.join(project_dir, 'references')
# Set up the output directory
output_dir = os.path.join(project_dir, 'results')

# Create the output directory if it doesn't exist
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

# Get the list of all voxel sizes
intersection_files = []
if legs == 'all' and voxel_sizes == 'all':
    intersection_files = glob.glob(os.path.join(valid_rays_dir, '*_intersections.parquet'))
elif legs == 'all' and isinstance(voxel_sizes, list):
    for voxel_size in voxel_sizes:
        intersection_files += glob.glob(os.path.join(valid_rays_dir, f'leg_*_voxel_{voxel_size}_intersections.parquet'))
elif isinstance(legs, list) and voxel_sizes == 'all':
    for leg in legs:
        intersection_files += glob.glob(os.path.join(valid_rays_dir, f'leg_{leg}_*_intersections.parquet'))
else:
    for leg in legs:
        for voxel_size in voxel_sizes:
            intersection_files += glob.glob(os.path.join(valid_rays_dir, f'leg_{leg}_voxel_{voxel_size}_intersections.parquet'))

# Check if any intersection files were found
if intersection_files == []:
    print("No intersection files found. Please check the input parameters.")

# Split intersection files into separate lists for each voxel_size
voxel_size_files = {}
for file in intersection_files:
    # Extract the voxel size from the filename
    parts = file.split('_')
    voxel_size = float(parts[parts.index('voxel') + 1])
    
    # Add the file to the corresponding voxel size list
    if voxel_size not in voxel_size_files:
        voxel_size_files[voxel_size] = []
    voxel_size_files[voxel_size].append(file)

# Extract voxel information for each voxel size
for voxel_size, files in voxel_size_files.items():
    # Create a list of all legs in files
    legs = []
    for file in files:
        leg = os.path.basename(file)
        parts = leg.split('_')
        leg = int(parts[parts.index('leg') + 1])
        legs.append(leg)

    # Calculate the lambda_1 for average leaf area
    lambda_1 = utils.calculate_lambda_1(voxel_size=voxel_size, average_leaf_area=average_leaf_area)
    print(f"Voxel size: {voxel_size}, Lambda_1: {lambda_1}")

    # Calculate per voxel information from all files
    voxel_metrics_df = utils.get_voxel_metrics(intersections_files=files, lambda_1=lambda_1, is_leaf_true=True)

    # Retrieve the reference file
    reference_file = glob.glob(os.path.join(references_dir, f'*voxel_size_{voxel_size}*'))[0]
    df_ref = pd.read_csv(reference_file)

    # CI_leaf_Corr, CI_lw_Corr
    # Ensure only numeric columns are included in the mean operation
    df_ref = df_ref.groupby('voxel_id').mean(numeric_only=True).reset_index()
    df_ref = df_ref.add_suffix('_ref')

    df_ref.rename(columns={
        'voxel_id_ref': 'voxel_id', 
        'voxel_cx_ref': 'voxel_cx',
        'voxel_cy_ref': 'voxel_cy',
        'voxel_cz_ref': 'voxel_cz',
        'LAD_ref_ref': 'LAD_ref', 
        'PAD_ref_ref': 'PAD_ref'
        }, inplace=True)

    # Merge to maintain voxel_id matching
    voxel_metrics_df = voxel_metrics_df.merge(df_ref, on='voxel_id', how='left')

    # Add lambda_1 to the dataframe
    voxel_metrics_df['lambda_1'] = lambda_1

    ### Add LAD calculations here if desired
    """Example, LAD_BL_TLS

    # Retrieve required variables
    I_leaf = voxel_metrics_df['I_leaf'].values
    mean_path_length = voxel_metrics_df['mean_path_length'].values  
    G_leaf = voxel_metrics_df['G_leaf'].values
    CI_leaf_ref = voxel_metrics_df['CI_leaf_corr_ref'].values

    LAD_BL_TLS = utils.BL_pimont_2018(I=I_leaf, mean_path_length=mean_path_length)
    LAD_BL_TLS_G = utils.BL_pimont_2018(I=I_leaf, mean_path_length=mean_path_length, G=G_leaf)
    LAD_BL_TLS_CI_ref = utils.BL_pimont_2018(I=I_leaf, mean_path_length=mean_path_length, G=G_leaf, CI=CI_leaf_ref)
    """

    # Save outputs to csv
    project_name = os.path.basename(os.path.normpath(project_dir))
    legs.sort()
    leg_string = "_".join(map(str, legs))
    output_file = os.path.join(output_dir, f"{project_name}_leg_{leg_string}_voxel_size_{voxel_size}.csv")
    if os.path.exists(output_file):
        os.remove(output_file)
    voxel_metrics_df.to_csv(output_file)

Voxel size: 1.5, Lambda_1: 0.005925925925925926
[########################################] | 100% Completed | 4.42 sms
